# 06 FTMO vs Standard - MA Cross

So sanh cung mot symbol/timeframe giua `standard` va `ftmo`. Neu standard dep nhung FTMO fail, chien luoc chua phu hop prop rules voi cau hinh hien tai.

In [ ]:
import sys
from pathlib import Path

def find_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / 'pyproject.toml').exists() and (p / 'core_python' / 'shared').exists():
            return p
    raise RuntimeError('Cannot find SEN05 repo root')

ROOT = find_root(Path.cwd())
CORE = ROOT / 'core_python'
for p in (str(ROOT), str(CORE)):
    if p not in sys.path:
        sys.path.insert(0, p)
print('ROOT =', ROOT)

In [ ]:
from IPython.display import display

from core_python.strategies.ma_cross.research_utils import configure_notebook, metric_row, metrics_frame, show_run_config, show_strategy_summary
from core_python.strategies.ma_cross.symbol.backtest import run_symbol_backtest

configure_notebook()
show_strategy_summary()

In [ ]:
RUN_CONFIG = {
    'symbol': 'US30',
    'tf': 'M30',
    'initial_balance': 100_000.0,
    'date_from': '2023-01-01',
    'date_to': None,
    'max_bars': 50_000,
    'indicator_overrides': {},
    'strategy_overrides': {},
    'costs': {},
    'broker_profile': None,
}
show_run_config('MA Cross account-mode comparison config', RUN_CONFIG)

In [ ]:
rows = []
results = {}
for mode in ['standard', 'ftmo']:
    result = run_symbol_backtest(
        RUN_CONFIG['symbol'],
        init_eq=RUN_CONFIG['initial_balance'],
        account_mode=mode,
        tf=RUN_CONFIG['tf'],
        date_from=RUN_CONFIG['date_from'],
        date_to=RUN_CONFIG['date_to'],
        max_bars=RUN_CONFIG['max_bars'],
        indicator_overrides=RUN_CONFIG['indicator_overrides'],
        strategy_overrides=RUN_CONFIG['strategy_overrides'],
        costs=RUN_CONFIG['costs'],
        broker_profile=RUN_CONFIG['broker_profile'],
    )
    results[mode] = result
    row = metric_row(result, symbol=RUN_CONFIG['symbol'], timeframe=RUN_CONFIG['tf'], params=RUN_CONFIG['indicator_overrides'], strategy_overrides=RUN_CONFIG['strategy_overrides'])
    row['account_mode'] = mode
    rows.append(row)

comparison = metrics_frame(rows)
display(comparison)

for mode, result in results.items():
    result.equity.rename(mode).plot(title='MA Cross standard vs FTMO equity', figsize=(12, 4), legend=True);